In [2]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

# 加载数据集
file_path = '关联规则_Online_Retail.xlsx'
df = pd.read_excel(file_path)

# 数据预处理
# 对于 Description 特征取值中的多余空格，采用 strip() 函数删除。
missing_values = df.isnull().sum()
missing_features = missing_values[missing_values > 0].index.tolist()
missing_countries = df[df.isnull().any(axis=1)]['Country'].unique()
# 对于Description特征取值中的多余空格，采用strip()函数删除。
df['Description'] = df['Description'].str.strip()
# 删除已取消的交易，即把InvoiceNo特征下取值以C开头的交易删除
df.dropna(subset=['CustomerID'], inplace=True)
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df = df[~df['InvoiceNo'].str.startswith('C')]
# Display the first few rows of the dataset
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536370,22728,ALARM CLOCK BAKELIKE PINK,24,2010-12-01 08:45:00,3.75,12583.0,France
1,536370,22727,ALARM CLOCK BAKELIKE RED,24,2010-12-01 08:45:00,3.75,12583.0,France
2,536370,22726,ALARM CLOCK BAKELIKE GREEN,12,2010-12-01 08:45:00,3.75,12583.0,France
3,536370,21724,PANDA AND BUNNIES STICKER SHEET,12,2010-12-01 08:45:00,0.85,12583.0,France
4,536370,21883,STARS GIFT TAPE,24,2010-12-01 08:45:00,0.65,12583.0,France


In [5]:
# 将数据集按国家划分为3个子集
# 并把每个子集转换为以InvoiceNo为索引的事务型数据集
# 使用自定义的hot_encode()函数，将事务数据编码为符合Apriori算法要求的0/1取值的数据。
df_france = df[df['Country'] == 'France']
df_portugal = df[df['Country'] == 'Portugal']
df_sweden = df[df['Country'] == 'Sweden']

# 定义热编码函数
def hot_encode(x):
    return x >= 1


# 转换为事务型数据集
basket_france = df_france.groupby(['InvoiceNo', 'Description'])['Quantity'].sum().unstack().reset_index().fillna(0).set_index('InvoiceNo')
basket_portugal = df_portugal.groupby(['InvoiceNo', 'Description'])['Quantity'].sum().unstack().reset_index().fillna(0).set_index('InvoiceNo')
basket_sweden = df_sweden.groupby(['InvoiceNo', 'Description'])['Quantity'].sum().unstack().reset_index().fillna(0).set_index('InvoiceNo')

# 应用热编码
basket_france_encoded = basket_france.applymap(hot_encode)
basket_portugal_encoded = basket_portugal.applymap(hot_encode)
basket_sweden_encoded = basket_sweden.applymap(hot_encode)
# Display the first few rows of the encoded transactional dataset for France
basket_france_encoded.head()

Description,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 EGG HOUSE PAINTED WOOD,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,12 PENCILS SMALL TUBE RED RETROSPOT,12 PENCILS SMALL TUBE SKULL,12 PENCILS TALL TUBE POSY,12 PENCILS TALL TUBE RED RETROSPOT,12 PENCILS TALL TUBE WOODLAND,...,WRAP SUKI AND FRIENDS,WRAP VINTAGE PETALS DESIGN,YELLOW COAT RACK PARIS FASHION,YELLOW GIANT GARDEN THERMOMETER,ZINC STAR T-LIGHT HOLDER,ZINC FOLKART SLEIGH BELLS,ZINC HERB GARDEN CONTAINER,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536370,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536852,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536974,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
537065,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
537463,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [6]:
# 对法国的事务数据应用 Apriori 算法
frequent_itemsets_france = apriori(basket_france_encoded, min_support=0.05, use_colnames=True)
rules_france = association_rules(frequent_itemsets_france, metric="confidence", min_threshold=0.2)

# 显示法国的关联规则
print("France:")
print(rules_france[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values(by='lift', ascending=False))

# 对葡萄牙的事务数据应用 Apriori 算法
frequent_itemsets_portugal = apriori(basket_portugal_encoded, min_support=0.05, use_colnames=True)
rules_portugal = association_rules(frequent_itemsets_portugal, metric="confidence", min_threshold=0.2)

# 显示葡萄牙的关联规则
print("\n\nPortugal:")
print(rules_portugal[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values(by='lift', ascending=False))

# 对瑞典的事务数据应用 Apriori 算法
frequent_itemsets_sweden = apriori(basket_sweden_encoded, min_support=0.05, use_colnames=True)
rules_sweden = association_rules(frequent_itemsets_sweden, metric="confidence", min_threshold=0.2)

# 显示瑞典的关联规则
print("\n\nSweden:")
print(rules_sweden[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values(by='lift', ascending=False))

France:
                                    antecedents  \
65               (PACK OF 6 SKULL PAPER PLATES)   
66                 (PACK OF 6 SKULL PAPER CUPS)   
17               (CHILDRENS CUTLERY DOLLY GIRL)   
18                 (CHILDRENS CUTLERY SPACEBOY)   
254                  (ALARM CLOCK BAKELIKE RED)   
..                                          ...   
150  (SPACEBOY LUNCH BOX, DOLLY GIRL LUNCH BOX)   
22                       (DOLLY GIRL LUNCH BOX)   
94                      (RED  HARMONICA IN BOX)   
102           (RETROSPOT TEA SET CERAMIC 11 PC)   
40                (LUNCH BAG DOLLY GIRL DESIGN)   

                                           consequents   support  confidence  \
65                        (PACK OF 6 SKULL PAPER CUPS)  0.051414    0.909091   
66                      (PACK OF 6 SKULL PAPER PLATES)  0.051414    0.800000   
17                        (CHILDRENS CUTLERY SPACEBOY)  0.064267    0.892857   
18                      (CHILDRENS CUTLERY DOLLY GIRL)  0.0